# Inner-speech EEG: fixed-fold overfitting diagnostics

This independent notebook implements the instructor's diagnostic plan. It evaluates each
subject separately, keeps **128 channels**, preserves the four original direction labels,
and uses your **previously saved outer and inner CV indices**. It does not regenerate folds,
add EEG filtering/ICA/VMD, reject extra trials, or begin channel reduction.

Run the data/plan checks first, then the memorization test and BASELINE_A. Enable the feature-count
and family experiments after inspecting those results. MRMR and freezing the later reduction
procedure are explicitly deferred until you have reviewed ANOVA.

The prior notebook's code is not embedded or executed. Existing wavelet/time-domain/regional/full
descriptors are supplied through verified **pre-selection feature exports**. Their absence is
reported as pending; they are not silently replaced by invented extractors. A new PSD-bandpower
control is implemented here as requested.

**Scientific correction to Phase 13:** outer-test results describe generalization. They do not
select k, C, a classifier or a feature family. Those choices are made inside inner CV. The
feature-family/k curves are exploratory comparisons; choosing a winner from them and then
claiming its same outer score as an independent confirmation would be selection bias.

Large train/test gaps can reflect excessive capacity, weak signal, unsuitable representations
or data problems. This notebook measures these possibilities; it does not promise a particular
accuracy or force results into a desired range.


## Setup and unchanged experiment settings

Edit the paths to your existing data and result exports. A Kaggle path can be used directly.
`SAVED_SPLITS_ROOT` must contain the split JSON files from the previous experiment.
Missing or mismatched plans stop execution instead of creating a different experiment.

The default derivative loader reads every available session for one subject and keeps every
saved inner-speech trial, including existing report-flagged trials. If your previous model input
had any additional fixed processing, choose `INPUT_MODE = "existing_input"` and export that
exact input as described at the end. This avoids accidentally changing the EEG preprocessing.

For released FIF files, MNE zero is cue onset; action is 0.5–3.0 s relative to that zero,
equivalent to 1.0–3.5 s from trial start. `ACTION_ENDPOINT` must match your previous experiment:
`exclusive` gives 640 samples at 256 Hz; `inclusive` gives 641. Neither choice is selected by performance.


In [ ]:
# Uncomment if packages are missing, then restart the kernel:
# %pip install numpy pandas scipy matplotlib "mne>=1.6" scikit-learn threadpoolctl
# Later, only when enabling MRMR:
# %pip install mrmr-selection


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

DATA_ROOT = Path("./ds003626/derivatives")
SAVED_SPLITS_ROOT = Path("./previous_results/splits")
EXISTING_INPUT_ROOT = Path("./model_input_exports")
FEATURE_EXPORT_ROOT = Path("./feature_exports")
OUTPUT_ROOT = Path("./overfitting_diagnostics_results")

INPUT_MODE = "derivatives"  # Or "existing_input" for your exact previous model input.
SUBJECTS = [1]              # Later: list(range(1, 11)); each is evaluated independently.
RANDOM_SEED = 42            # Must agree with the saved plan.
CHANNEL_COUNT = 128
ACTION_WINDOW_MNE = (0.5, 3.0)
ACTION_ENDPOINT = "exclusive"  # Match the previous experiment; do not change it to tune accuracy.

C_VALUES = [0.001, 0.01, 0.1, 1.0]
K_VALUES = [10, 20, 40, 80, 120, 200]
MRMR_K_VALUES = [20, 40, 80, 120]
VARIANCE_THRESHOLD = 0.0  # Remove constants. Positive cutoffs require feature-unit justification.
CORRELATION_THRESHOLD = 0.95
INCLUDE_ALL_FEATURES_REFERENCE = False  # Enable later for the optional full-dimensional reference.
N_PERMUTATIONS = 5        # A small initial control, not a formal significance test.
N_JOBS = 1              # Conservative memory/CPU usage.
RESUME = True
ACTIVE_RUN_IDS = set()  # Populate only from experiments evaluated/reused by this execution.

RUN_MEMORIZATION = True
RUN_BASELINE_A = True
RUN_FEATURE_COUNT_SWEEP = False
RUN_FAMILY_COMPARISON = False
RUN_REDUNDANCY = True
RUN_PERMUTATION_CONTROL = False
RUN_NESTED_PROCEDURE_SELECTION = False
ANOVA_REVIEWED = False
RUN_MRMR = False
FREEZE_PROCEDURE = False

assert CHANNEL_COUNT == 128, "Channel reduction is deferred."
assert INPUT_MODE in {"derivatives", "existing_input"}
assert ACTION_ENDPOINT in {"exclusive", "inclusive"}
assert SUBJECTS and len(SUBJECTS) == len(set(SUBJECTS))
assert set(SUBJECTS).issubset(set(range(1, 11)))
assert not RUN_MRMR or ANOVA_REVIEWED, "Review the ANOVA results before enabling MRMR."

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("scikit-learn:", sklearn.__version__)
print("Subjects:", SUBJECTS, "| fixed channel count:", CHANNEL_COUNT, "| seed:", RANDOM_SEED)


## Reusable data checks and frozen-plan handling

The split reader requires the saved structure `seed`, `trial_ids`, `labels`, `folds`.
Each fold has global `train`/`test` indices and an `inner` list of global `train`/`validation`
indices. Indices are mapped to outer-training-local coordinates before GridSearchCV.

The checks enforce exact trial/label identity, disjoint partitions, four classes per split,
one outer OOF prediction per original trial, and zero outer-test access in inner CV.
Local files are read only. Output checks and experiment CSVs go to OUTPUT_ROOT.


In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

CORE_VERSION = "fixed-fold-overfitting-diagnostics-1.0"
CLASS_NAMES = {0: "Up", 1: "Down", 2: "Right", 3: "Left"}
FAMILY_NAMES = ("bandpower", "wavelet", "time_domain", "regional", "full")
CORE_SOURCE_SHA256 = 'd1ff582f98c93ca4a6bc5aab33748d83dd54825715e7ae84e5cf7a3d626a64f1'

def digest_json(value):
    text = json.dumps(value, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(text.encode()).hexdigest()

def array_digest(array):
    array = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str((array.shape, array.dtype.str)).encode())
    view = memoryview(array).cast("B")
    for start in range(0, len(view), 8 * 1024 * 1024):
        digest.update(view[start:start + 8 * 1024 * 1024])
    return digest.hexdigest()

def save_csv_atomic(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def save_json_atomic(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def load_derivative_subject(data_root, subject, action_window=(0.5, 3.0), endpoint="exclusive"):
    """Use all saved inner trials, all EEG channels, and no additional EEG processing."""
    import mne
    root = Path(data_root)
    subject_name = f"sub-{int(subject):02d}"
    files = sorted(root.rglob(f"{subject_name}_ses-*_eeg-epo.fif"))
    if not files:
        raise FileNotFoundError(f"No derivative EEG files for {subject_name} below {root}")
    parts, metadata_parts, source_rows = [], [], []
    names_reference = None
    times_reference = None
    sampling_reference = None
    seen_sessions = set()
    for eeg_path in files:
        session_name = eeg_path.name.split("_")[1]
        if session_name in seen_sessions:
            raise ValueError("Duplicate recording: DATA_ROOT must point at one dataset version.")
        seen_sessions.add(session_name)
        event_path = eeg_path.with_name(eeg_path.name.replace("_eeg-epo.fif", "_events.dat"))
        events = np.asarray(pd.read_pickle(event_path), dtype=np.int64)
        epochs = mne.read_epochs(eeg_path, preload=False, proj=False, verbose="ERROR")
        if events.ndim != 2 or events.shape != (len(epochs.events), 4):
            raise ValueError(f"Incorrect event shape for {eeg_path.name}")
        if not np.array_equal(events[:, 0], epochs.events[:, 0]):
            raise ValueError("EEG/label timestamp alignment failed.")
        if not np.array_equal(events[:, 1] + 31, epochs.events[:, 2]):
            raise ValueError("EEG/label class alignment failed.")
        if not np.isin(events[:, 1], [0, 1, 2, 3]).all():
            raise ValueError("Unexpected direction labels.")
        original_rows = np.asarray(epochs.selection, dtype=int)
        selected = np.flatnonzero(events[:, 2] == 1)
        epochs = epochs[selected].load_data()
        epochs.pick(mne.pick_types(epochs.info, eeg=True, exclude=[]))
        if len(epochs.ch_names) != 128:
            raise ValueError("This debugging experiment requires exactly 128 EEG channels.")
        start, end = action_window
        mask = (epochs.times >= start) & (
            (epochs.times < end) if endpoint == "exclusive" else (epochs.times <= end)
        )
        if not mask.any():
            raise ValueError("The declared action window contains no samples.")
        names = list(epochs.ch_names)
        times = epochs.times[mask]
        sampling = float(epochs.info["sfreq"])
        if names_reference is not None:
            if names != names_reference or sampling != sampling_reference or not np.array_equal(times, times_reference):
                raise ValueError("Channels or time coordinates differ across sessions.")
        names_reference, times_reference, sampling_reference = names, times, sampling
        signal = epochs.get_data(copy=False)[:, :, mask].astype(np.float32)
        if len(signal) != len(selected):
            raise ValueError("Loading unexpectedly changed trial count.")
        parts.append(signal)
        session_number = int(session_name.replace("ses-", ""))
        metadata_parts.append(pd.DataFrame({
            "subject": int(subject), "session": session_number,
            "trial_index": original_rows[selected], "cue_sample_raw": events[selected, 0],
            "class_id": events[selected, 1],
            "trial_id": [f"{subject_name}_{session_name}_row-{row:04d}" for row in original_rows[selected]],
        }))
        source_rows.append({"eeg": str(eeg_path), "events": str(event_path), "inner_trials": len(selected)})
    X = np.concatenate(parts)
    metadata = pd.concat(metadata_parts, ignore_index=True)
    provenance = {
        "mode": "derivatives", "units": "volts", "channels": names_reference,
        "action_window_mne": list(action_window), "endpoint": endpoint,
        "extra_eeg_processing": False, "extra_trial_rejection": False,
        "sources": source_rows, "X_sha256": array_digest(X),
    }
    return X, metadata, sampling_reference, names_reference, times_reference, provenance

def load_existing_input(input_root, subject):
    """Preferred when the previous model input includes additional fixed processing."""
    prefix = Path(input_root) / f"sub-{int(subject):02d}"
    manifest = json.loads(prefix.with_suffix(".json").read_text(encoding="utf-8"))
    with np.load(prefix.with_suffix(".npz"), allow_pickle=False) as bundle:
        X = np.asarray(bundle["X"], dtype=np.float32)
        times = np.asarray(bundle["times"], dtype=float)
        trial_ids = np.asarray(bundle["trial_ids"]).astype(str)
        labels = np.asarray(bundle["y"], dtype=int)
        sessions = np.asarray(bundle["sessions"], dtype=int)
        names = np.asarray(bundle["ch_names"]).astype(str).tolist()
    if manifest["units"] == "microvolts":
        X = X * 1e-6
    elif manifest["units"] != "volts":
        raise ValueError("Declare existing input units as volts or microvolts.")
    if not manifest.get("inner_speech_only", False):
        raise ValueError("The input export must explicitly contain inner-speech trials only.")
    if X.ndim != 3 or X.shape[1] != 128 or X.shape[2] != len(times):
        raise ValueError("Input must be [trials, 128 channels, time points].")
    metadata = pd.DataFrame({"subject": int(subject), "session": sessions,
                             "class_id": labels, "trial_id": trial_ids})
    provenance = dict(manifest, mode="existing_input", X_sha256=array_digest(X))
    return X, metadata, float(manifest["sfreq"]), names, times, provenance

def validate_input(X, metadata, ch_names):
    if X.ndim != 3 or X.shape[1] != 128 or len(ch_names) != 128:
        raise ValueError("Exactly 128 channels are required; no reduction in this notebook.")
    if len(X) != len(metadata) or not metadata.trial_id.is_unique:
        raise ValueError("Trial identity/row count check failed.")
    if set(metadata.class_id) != set(CLASS_NAMES) or metadata.subject.nunique() != 1:
        raise ValueError("Load one subject with all four direction classes.")
    n_nan = sum(int(np.isnan(trial).sum()) for trial in X)
    n_inf = sum(int(np.isinf(trial).sum()) for trial in X)
    print("X.shape:", X.shape)
    print("y.shape:", metadata.class_id.to_numpy().shape)
    print("np.unique(y, return_counts=True):", np.unique(metadata.class_id, return_counts=True))
    print("NaN values:", n_nan, "Inf values:", n_inf)
    if n_nan or n_inf:
        raise ValueError("Inspect nonfinite data; this notebook does not add imputation or rejection.")

def find_saved_plan(split_root, subject, seed, metadata):
    root = Path(split_root)
    possible = sorted(root.rglob(f"sub-{int(subject):02d}*.json"))
    matches = []
    current_ids = set(metadata.trial_id.astype(str))
    for path in possible:
        plan = json.loads(path.read_text(encoding="utf-8"))
        if "folds" not in plan or "trial_ids" not in plan:
            continue
        if int(plan.get("seed", -1)) == seed and set(map(str, plan["trial_ids"])) == current_ids:
            matches.append((path, plan))
    if not matches:
        raise FileNotFoundError(
            f"No saved nested-CV plan matches subject {subject}, seed {seed}, and these exact trials. "
            "Set SAVED_SPLITS_ROOT to the previous run's splits directory. Folds are never regenerated here."
        )
    signatures = {digest_json(plan) for _, plan in matches}
    if len(signatures) != 1:
        raise ValueError("Multiple different saved plans match. Point at one experiment's split directory.")
    return matches[0]

def align_and_validate_plan(X, metadata, plan, seed):
    if int(plan["seed"]) != seed:
        raise ValueError("Saved seed differs from RANDOM_SEED.")
    ids = list(map(str, plan["trial_ids"]))
    if len(ids) != len(set(ids)) or set(ids) != set(metadata.trial_id.astype(str)):
        raise ValueError("Saved trials differ from loaded trials; no silent dropping or intersection.")
    lookup = {str(value): i for i, value in enumerate(metadata.trial_id)}
    order = np.array([lookup[value] for value in ids], dtype=int)
    X = X[order]
    metadata = metadata.iloc[order].reset_index(drop=True)
    y = metadata.class_id.to_numpy(dtype=int)
    if not np.array_equal(y, np.asarray(plan["labels"], dtype=int)):
        raise ValueError("Saved labels differ from the EEG event labels.")
    universe = set(range(len(y)))
    coverage = np.zeros(len(y), dtype=int)
    fold_ids = [int(fold.get("fold", i)) for i, fold in enumerate(plan["folds"], 1)]
    if len(fold_ids) != len(set(fold_ids)):
        raise ValueError("Outer fold identifiers must be unique.")
    for fold in plan["folds"]:
        train, test = np.asarray(fold["train"], dtype=int), np.asarray(fold["test"], dtype=int)
        if len(set(train)) != len(train) or len(set(test)) != len(test):
            raise ValueError("Duplicate indices inside an outer split.")
        if set(train) & set(test) or set(train) | set(test) != universe:
            raise ValueError("Outer train/test must be disjoint and cover all original trials.")
        if set(y[train]) != set(CLASS_NAMES) or set(y[test]) != set(CLASS_NAMES):
            raise ValueError("An outer split lacks a class.")
        coverage[test] += 1
        inner_coverage = {int(index): 0 for index in train}
        for inner in fold["inner"]:
            inner_train, validation = set(inner["train"]), set(inner["validation"])
            if len(inner_train) != len(inner["train"]) or len(validation) != len(inner["validation"]):
                raise ValueError("Duplicate indices inside an inner split.")
            if inner_train & validation or inner_train | validation != set(train):
                raise ValueError("Saved inner indices must be global indices partitioning outer training.")
            if (inner_train | validation) & set(test):
                raise ValueError("Outer-test indices appeared inside inner CV.")
            if set(y[list(inner_train)]) != set(CLASS_NAMES) or set(y[list(validation)]) != set(CLASS_NAMES):
                raise ValueError("An inner split lacks a class.")
            for index in validation:
                inner_coverage[int(index)] += 1
        if set(inner_coverage.values()) != {1}:
            raise ValueError("Saved inner validation folds must cover each outer-training trial once.")
    if not np.all(coverage == 1):
        raise ValueError("Saved outer folds must predict every original trial exactly once.")
    return X, y, metadata

def inner_cv_local(fold):
    lookup = {int(index): i for i, index in enumerate(fold["train"])}
    return [(np.array([lookup[int(i)] for i in inner["train"]]),
             np.array([lookup[int(i)] for i in inner["validation"]])) for inner in fold["inner"]]


## Feature inputs: the new control and your existing feature families

BASELINE_A has five log-bandpowers per EEG channel: delta 1–4, theta 4–8, alpha 8–13,
beta 13–30 and gamma 30–45 Hz. This gives 640 descriptors before constant removal.
Welch and fixed band integration operate independently on each trial; no across-trial
mean, ranking, labels or fitted projection enter extraction.

The other four families must be raw exports from your current extractor. They must be
computed independently per trial, before any scaling, ANOVA, PCA, MRMR or across-trial
normalization. Globally fitted feature exports are rejected. If your extractor learns a
projection from several trials, exporting its final matrix is insufficient: that learned
extractor must instead be fitted within folds in a separate adapter before using it here.

The checksum checks input identity, not the truth of the provenance declaration. Inspect
the original export step before declaring `fit_scope = "stateless_per_trial"`.


In [ ]:
def bandpower_features(X, sfreq):
    """Fixed per-trial Welch/log-bandpower; no cross-trial estimation."""
    bands = ((1., 4.), (4., 8.), (8., 13.), (13., 30.), (30., 45.))
    nperseg = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or nperseg < 16:
        raise ValueError("Signal sampling/window cannot support the requested PSD bands.")
    freqs, density = welch(X, fs=sfreq, nperseg=nperseg,
                          noverlap=nperseg // 2, axis=-1, scaling="density")
    values = []
    for low, high in bands:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError("Too few frequency bins for band integration.")
        power = np.trapezoid(density[..., mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def load_existing_families(feature_root, subject, metadata, provenance):
    """Only accept existing, unselected, independently computed per-trial descriptors."""
    prefix = Path(feature_root) / f"sub-{int(subject):02d}"
    if not prefix.with_suffix(".npz").exists():
        print("Existing feature export absent: wavelet/time/regional/full experiments remain pending.")
        return {}, {"status": "not_available"}
    manifest = json.loads(prefix.with_suffix(".json").read_text(encoding="utf-8"))
    if manifest.get("fit_scope") != "stateless_per_trial":
        raise ValueError("Export raw per-trial descriptors, not globally scaled/ANOVA/PCA/MRMR-selected features.")
    if manifest.get("model_input_sha256") != provenance["X_sha256"]:
        raise ValueError("Feature export does not match this exact model-input array/order/units.")
    if int(manifest.get("channel_count", -1)) != 128:
        raise ValueError("Feature export must represent the unchanged 128-channel configuration.")
    families = {}
    with np.load(prefix.with_suffix(".npz"), allow_pickle=False) as bundle:
        ids = np.asarray(bundle["trial_ids"]).astype(str)
        labels = np.asarray(bundle["y"], dtype=int)
        if len(ids) != len(set(ids)) or set(ids) != set(metadata.trial_id.astype(str)):
            raise ValueError("Feature export trial identities differ.")
        lookup = {value: i for i, value in enumerate(ids)}
        order = np.array([lookup[value] for value in metadata.trial_id.astype(str)])
        if not np.array_equal(labels[order], metadata.class_id.to_numpy()):
            raise ValueError("Feature-export labels are misaligned.")
        for family in FAMILY_NAMES[1:]:
            if family not in bundle:
                continue
            matrix = np.asarray(bundle[family], dtype=np.float64)[order]
            if matrix.ndim != 2 or len(matrix) != len(metadata) or not matrix.shape[1]:
                raise ValueError(f"Invalid matrix for {family}")
            if not np.isfinite(matrix).all():
                raise ValueError(f"Nonfinite values in existing {family} features.")
            families[family] = matrix
    return families, manifest

def combine_family_inputs(families):
    groups, chunks, offset = {}, [], 0
    for name, matrix in families.items():
        if matrix.ndim != 2 or not np.isfinite(matrix).all():
            raise ValueError(f"Invalid or nonfinite features in {name}")
        groups[name] = list(range(offset, offset + matrix.shape[1]))
        chunks.append(matrix)
        offset += matrix.shape[1]
    return np.concatenate(chunks, axis=1), groups


## Fold-local selection and low-capacity classifiers

Every GridSearchCV candidate includes family selection, variance filtering, StandardScaler,
ANOVA or optional MRMR, and a linear classifier. All learned steps receive only training rows.
The grid fits **LinearSVC** and **LogisticRegression**, with C = 0.001, 0.01, 0.1, 1.0.
No RBF SVM is used for generalization experiments.

The ANOVA wrapper uses sklearn SelectKBest. When variance filtering leaves fewer features
than requested k, it caps k and records the actual selected count. Numerical inner-score ties
favor fewer features and stronger regularization. These decisions use no outer-test scores.

MRMR is optional and deferred. Its implementation uses the mrmr-selection package's
F-statistic relevance and correlation redundancy; it is not presented as an MI/MI variant.


In [ ]:
class FamilyColumns(BaseEstimator, TransformerMixin):
    def __init__(self, groups, name="bandpower"):
        self.groups = groups
        self.name = name

    def fit(self, X, y=None):
        self.indices_ = np.asarray(self.groups[self.name], dtype=int)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.indices_]

class BoundedANOVA(BaseEstimator, TransformerMixin):
    def __init__(self, k=40):
        self.k = k

    def fit(self, X, y):
        self.n_features_in_ = X.shape[1]
        self.k_effective_ = self.k if self.k == "all" else min(int(self.k), X.shape[1])
        self.selector_ = SelectKBest(f_classif, k=self.k_effective_).fit(X, y)
        return self

    def transform(self, X):
        return self.selector_.transform(X)

    def get_support(self):
        return self.selector_.get_support()

class FoldMRMR(BaseEstimator, TransformerMixin):
    """Optional mrmr-selection wrapper; fit receives training rows only."""
    def __init__(self, k=40):
        self.k = k

    def fit(self, X, y):
        from mrmr import mrmr_classif
        self.n_features_in_ = X.shape[1]
        self.k_effective_ = min(int(self.k), X.shape[1])
        columns = [f"f{i}" for i in range(X.shape[1])]
        chosen = mrmr_classif(X=pd.DataFrame(X, columns=columns), y=pd.Series(np.asarray(y)),
                             K=self.k_effective_, relevance="f", redundancy="c",
                             n_jobs=1, show_progress=False)
        self.indices_ = np.array([int(name[1:]) for name in chosen])
        if not len(self.indices_):
            raise ValueError("MRMR selected no usable features.")
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.indices_]

    def get_support(self):
        mask = np.zeros(self.n_features_in_, dtype=bool)
        mask[self.indices_] = True
        return mask

def make_estimator(groups, family, seed, selector="anova", variance_threshold=0.0):
    return Pipeline([
        ("family", FamilyColumns(groups, name=family)),
        ("variance", VarianceThreshold(threshold=variance_threshold)),
        ("scaler", StandardScaler()),
        ("selector", BoundedANOVA() if selector == "anova" else FoldMRMR()),
        ("model", LinearSVC(dual="auto", max_iter=20000, random_state=seed)),
    ])

def make_grid(families, counts, c_values, seed, classifiers=("linear_svm", "logistic")):
    grids = []
    for classifier in classifiers:
        if classifier == "linear_svm":
            model = LinearSVC(dual="auto", max_iter=20000, random_state=seed)
        elif classifier == "logistic":
            model = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=seed)
        else:
            raise ValueError("Use only linear_svm/logistic for the generalization experiment.")
        grids.append({"family__name": list(families), "selector__k": list(counts),
                      "model": [model], "model__C": list(c_values)})
    return grids

def simple_refit_index(cv_results):
    """Inner BA first; numerical ties favor fewer features, then stronger regularization."""
    scores = np.asarray(cv_results["mean_test_score"], dtype=float)
    best = np.nanmax(scores)
    candidates = np.flatnonzero(np.isclose(scores, best, rtol=0, atol=1e-12))
    def tie_key(index):
        params = cv_results["params"][index]
        k = params["selector__k"]
        count = float("inf") if k == "all" else int(k)
        return count, float(params["model__C"]), int(index)
    return int(min(candidates, key=tie_key))


## Persistent train/test diagnostics and CSV checkpoints

Every completed outer fold records subject, channel count, feature family, selected count,
classifier, fold, C, seed, train BA, outer-test BA and their gap. Training BA is a resubstitution
diagnostic, not independent performance. The inner search results, warnings, OOF predictions
and experiment fingerprints are also saved. CSVs are updated atomically after each fold.

Resume keys include feature/input values, labels, saved splits, search settings and the core
version. Experiments with different data or settings are not silently pooled as one result.
If you manually edit a helper implementation, change CORE_VERSION or set RESUME=False.


In [ ]:
def append_checkpoint(rows, predictions, output_root):
    root = Path(output_root)
    result_path, oof_path = root / "fold_results.csv", root / "oof_predictions.csv"
    fresh = pd.DataFrame(rows)
    if result_path.exists():
        fresh = pd.concat([pd.read_csv(result_path), fresh], ignore_index=True)
    fresh = fresh.drop_duplicates(["run_id", "outer_fold"], keep="last")
    save_csv_atomic(fresh, result_path)
    oof = pd.DataFrame(predictions)
    if oof_path.exists():
        oof = pd.concat([pd.read_csv(oof_path), oof], ignore_index=True)
    oof = oof.drop_duplicates(["run_id", "trial_id"], keep="last")
    save_csv_atomic(oof, oof_path)

def evaluate_nested(F, groups, y, metadata, plan, subject, experiment, families,
                    counts, c_values, seed, output_root, selector="anova",
                    classifiers=("linear_svm", "logistic"), variance_threshold=0.0,
                    permutation_id=0, resume=True, n_jobs=1):
    """All learned operations are fitted by GridSearchCV on outer training only."""
    model_identity = dict(core_version=CORE_VERSION, subject=subject, experiment=experiment,
                          core_source_sha256=globals().get("CORE_SOURCE_SHA256", CORE_VERSION),
                          families=list(families), counts=list(counts), Cs=list(c_values),
                          seed=seed, selector=selector, classifiers=list(classifiers),
                          variance_threshold=variance_threshold, permutation_id=permutation_id,
                          splits=digest_json(plan), features=array_digest(F), labels=array_digest(y),
                          n_trials=len(y), expected_trial_ids=metadata.trial_id.astype(str).tolist(),
                          expected_fold_ids=[int(f.get("fold", i)) for i, f in enumerate(plan["folds"], 1)],
                          sklearn_version=__import__("sklearn").__version__,
                          groups_signature=digest_json(groups))
    run_id = digest_json(model_identity)[:24]
    root = Path(output_root)
    save_json_atomic(model_identity, root / "run_manifests" / f"{run_id}.json")
    previous_path = root / "fold_results.csv"
    previous = pd.read_csv(previous_path) if previous_path.exists() else pd.DataFrame()
    oof_path = root / "oof_predictions.csv"
    old_predictions = pd.read_csv(oof_path) if oof_path.exists() else pd.DataFrame()
    rows = []
    for ordinal, fold in enumerate(plan["folds"], start=1):
        fold_id = int(fold.get("fold", ordinal))
        train = np.asarray(fold["train"], dtype=int)
        test = np.asarray(fold["test"], dtype=int)
        if resume and len(previous) and len(old_predictions):
            done = previous.loc[(previous.run_id == run_id) & (previous.outer_fold == fold_id)]
            old = old_predictions.loc[(old_predictions.run_id == run_id) & old_predictions.trial_id.isin(metadata.trial_id.iloc[test])]
            if len(done) == 1 and len(old) == len(test):
                rows.append(done.iloc[0].to_dict())
                continue
        y_train = y[train].copy()
        if permutation_id:
            # Shuffle only outer-training labels, within session; outer-test truth is fixed.
            rng = np.random.default_rng(seed + 100000 * permutation_id + 1000 * subject + fold_id)
            sessions = metadata.session.to_numpy()[train]
            for session in np.unique(sessions):
                positions = np.flatnonzero(sessions == session)
                y_train[positions] = rng.permutation(y_train[positions])
        estimator = make_estimator(groups, families[0], seed, selector, variance_threshold)
        search = GridSearchCV(
            estimator, make_grid(families, counts, c_values, seed, classifiers),
            scoring="balanced_accuracy", cv=inner_cv_local(fold),
            refit=simple_refit_index, return_train_score=True,
            n_jobs=n_jobs, error_score="raise",
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            with threadpool_limits(limits=1):
                search.fit(F[train], y_train)
                model = search.best_estimator_
                train_prediction = model.predict(F[train])
                test_prediction = model.predict(F[test])
        train_ba = float(balanced_accuracy_score(y_train, train_prediction))
        test_ba = float(balanced_accuracy_score(y[test], test_prediction))
        steps = model.named_steps
        classifier_name = "linear_svm" if isinstance(steps["model"], LinearSVC) else "logistic"
        selected_count = int(steps["selector"].get_support().sum())
        family = steps["family"].name
        warning_messages = sorted({str(w.message) for w in caught})
        row = dict(
            subject=subject, channel_count=128, experiment=experiment,
            feature_family=family, selector=selector,
            num_features_original=len(groups[family]), num_features=selected_count,
            requested_k=str(search.best_params_["selector__k"]), classifier=classifier_name,
            outer_fold=fold_id, train_balanced_accuracy=train_ba,
            test_balanced_accuracy=test_ba, generalization_gap=train_ba - test_ba,
            best_C=float(search.best_params_["model__C"]), random_seed=seed,
            inner_best_balanced_accuracy=float(search.cv_results_["mean_test_score"][search.best_index_]),
            permutation_id=permutation_id,
            train_label_basis="shuffled" if permutation_id else "original",
            n_train=len(train), n_test=len(test), run_id=run_id,
            split_signature=digest_json(plan), convergence_warning=any("converg" in message.lower() for message in warning_messages),
        )
        predictions = [dict(run_id=run_id, subject=subject, experiment=experiment,
                            outer_fold=fold_id, trial_id=str(metadata.trial_id.iloc[index]),
                            y_true=int(y[index]), y_pred=int(prediction),
                            permutation_id=permutation_id)
                       for index, prediction in zip(test, test_prediction)]
        append_checkpoint([row], predictions, root)
        save_csv_atomic(pd.DataFrame(search.cv_results_), root / "inner_search" / f"{run_id}_fold-{fold_id}.csv")
        save_json_atomic(warning_messages, root / "warnings" / f"{run_id}_fold-{fold_id}.json")
        print(f"Subject {subject:02d} Fold {fold_id} | {experiment} | {family} | {classifier_name}\n"
              f"Features: {selected_count} (requested {row['requested_k']}) | C: {row['best_C']}\n"
              f"Train BA: {train_ba:.3f} | Test BA: {test_ba:.3f} | Gap: {train_ba - test_ba:.3f}")
        rows.append(row)
    return pd.DataFrame(rows)

def memorization_test(F, y, metadata, plan, seed, subject, output_root, requested_trials=60):
    train = np.asarray(plan["folds"][0]["train"], dtype=int)
    n_trials = min(requested_trials, len(train))
    if n_trials < len(train):
        sampler = StratifiedShuffleSplit(n_splits=1, train_size=n_trials, random_state=seed)
        selected, _ = next(sampler.split(F[train], y[train]))
        indices = train[selected]
    else:
        indices = train
    F_small, y_small = F[indices], y[indices]
    print("Memorization features X.shape:", F_small.shape, "y.shape:", y_small.shape)
    print("np.unique(y, return_counts=True):", np.unique(y_small, return_counts=True))
    print("NaN:", int(np.isnan(F_small).sum()), "Inf:", int(np.isinf(F_small).sum()))
    unique_rows, reverse = np.unique(F_small, axis=0, return_inverse=True)
    conflicts = sum(len(set(y_small[reverse == index])) > 1 for index in range(len(unique_rows)))
    print("Duplicate feature rows:", len(F_small) - len(unique_rows),
          "identical feature rows with conflicting labels:", conflicts)
    model = DecisionTreeClassifier(random_state=seed, min_samples_leaf=1)
    model.fit(F_small, y_small)
    prediction = model.predict(F_small)
    result = dict(subject=subject, trials=len(indices), features=F.shape[1],
                  training_accuracy=float(accuracy_score(y_small, prediction)),
                  training_balanced_accuracy=float(balanced_accuracy_score(y_small, prediction)),
                  duplicate_rows=len(F_small) - len(unique_rows), conflicting_duplicate_groups=conflicts,
                  diagnostic_only=True, original_fold=plan["folds"][0].get("fold", 1))
    save_json_atomic(result, Path(output_root) / "memorization" / f"sub-{subject:02d}.json")
    print(result)
    return result

def redundancy_diagnostic(F, feature_names, train_indices, output_root, subject,
                          family, fold_id, correlation_threshold=0.95,
                          variance_threshold=0.0, block_size=256):
    """Count all correlated feature pairs without building a full p-by-p matrix."""
    training = np.asarray(F[train_indices], dtype=np.float64)
    variances = training.var(axis=0)
    feature_names = np.asarray(feature_names, dtype=str)
    variance_table = pd.DataFrame({"feature": feature_names, "variance": variances,
                                   "removed_by_threshold": variances <= variance_threshold})
    selector = VarianceThreshold(variance_threshold).fit(training)
    training = selector.transform(training)
    kept_names = feature_names[selector.get_support()]
    centered = training - training.mean(axis=0)
    norms = np.linalg.norm(centered, axis=0)
    normalized = centered / norms
    count, example_pairs = 0, []
    for start in range(0, normalized.shape[1], block_size):
        end = min(start + block_size, normalized.shape[1])
        correlations = normalized[:, start:end].T @ normalized
        rows, columns = np.where(np.abs(correlations) >= correlation_threshold)
        valid = columns > (rows + start)
        count += int(valid.sum())
        for row, column in zip(rows[valid][:max(0, 50-len(example_pairs))], columns[valid][:max(0, 50-len(example_pairs))]):
            example_pairs.append({"feature_a": kept_names[start + row], "feature_b": kept_names[column],
                                  "correlation": float(correlations[row, column])})
    summary = dict(subject=subject, outer_fold=fold_id, feature_family=family,
                   original_features=F.shape[1], retained_after_variance=int(selector.get_support().sum()),
                   removed_near_zero=int((variances <= variance_threshold).sum()),
                   absolute_correlation_threshold=correlation_threshold,
                   strongly_correlated_pairs=count, training_rows=len(train_indices),
                   evaluated_pair_count=int(training.shape[1] * (training.shape[1]-1) // 2))
    root = Path(output_root) / "redundancy"
    tag = f"sub-{subject:02d}_{family}_fold-{fold_id}"
    save_csv_atomic(variance_table, root / f"{tag}_variance.csv")
    save_csv_atomic(pd.DataFrame(example_pairs), root / f"{tag}_example_pairs.csv")
    save_json_atomic(summary, root / f"{tag}_summary.json")
    print(summary)
    return summary


## Phase 1–2: prepare one subject and inspect the leakage boundary

`prepare_subject` never combines participants. It aligns data to the exact saved trial order
and builds the family registry. Feature-family blocks are kept separate in every candidate;
concatenation below is only a transport format for the family selector, not feature fusion.

The printed feature counts and p/n ratios help expose high dimensionality. The list below
checks this notebook; it does not retroactively prove the previous pipeline was leakage-free.


In [ ]:
def prepare_subject(subject):
    if INPUT_MODE == "derivatives":
        X, metadata, sfreq, ch_names, times, provenance = load_derivative_subject(
            DATA_ROOT, subject, ACTION_WINDOW_MNE, ACTION_ENDPOINT)
    else:
        X, metadata, sfreq, ch_names, times, provenance = load_existing_input(EXISTING_INPUT_ROOT, subject)
    validate_input(X, metadata, ch_names)
    split_path, plan = find_saved_plan(SAVED_SPLITS_ROOT, subject, RANDOM_SEED, metadata)
    X, y, trial_metadata = align_and_validate_plan(X, metadata, plan, RANDOM_SEED)
    families = {"bandpower": bandpower_features(X, sfreq)}
    existing, feature_manifest = load_existing_families(FEATURE_EXPORT_ROOT, subject, trial_metadata, provenance)
    families.update(existing)
    F, family_groups = combine_family_inputs(families)
    feature_count_table = pd.DataFrame({
        "feature_family": list(families),
        "original_features": [matrix.shape[1] for matrix in families.values()],
    })
    first_train_n = len(plan["folds"][0]["train"])
    feature_count_table["features_per_first_outer_training_trial"] = feature_count_table.original_features / first_train_n
    print("Frozen plan:", split_path)
    print("Outer folds:", len(plan["folds"]), "| first outer-training trials:", first_train_n)
    display(feature_count_table)
    audit = {
        "subject": subject, "input": provenance,
        "saved_plan_path": str(split_path), "split_signature": digest_json(plan),
        "trial_order_signature": digest_json(trial_metadata.trial_id.tolist()),
        "feature_export_provenance": feature_manifest,
        "fit_boundary": "GridSearchCV.fit receives outer-training rows only",
        "constant_filter": "inside every inner/refit Pipeline",
        "scaler": "inside every inner/refit Pipeline",
        "ANOVA_MRMR": "inside every inner/refit Pipeline",
        "model_family_k_C_selection": "inner CV only",
        "outer_test": "predict and score only",
        "extra_EEG_preprocessing": False if INPUT_MODE == "derivatives" else "unchanged exported input",
    }
    save_json_atomic(audit, OUTPUT_ROOT / "audits" / f"sub-{subject:02d}.json")
    return dict(subject=subject, X=X, y=y, trial_metadata=trial_metadata, sfreq=sfreq,
                ch_names=ch_names, times=times, provenance=provenance, plan=plan,
                families=families, F=F, family_groups=family_groups)


# Start with one subject so you can inspect the checks before a long run.
subject_data = prepare_subject(SUBJECTS[0])
X = subject_data["X"]
y = subject_data["y"]
trial_metadata = subject_data["trial_metadata"]
sfreq = subject_data["sfreq"]
ch_names = subject_data["ch_names"]
times = subject_data["times"]
fixed_plan = subject_data["plan"]

print("Input fingerprint for matching feature exports:", subject_data["provenance"]["X_sha256"])
display(trial_metadata.head())


## Phase 3–4: small memorization test

Take up to 60 stratified trials from **the first outer-training partition**, not the outer test.
A fully grown decision tree is used only to check memorability. High same-sample accuracy is
expected if rows are distinct and usable. If fewer than 40 training rows are available, the
actual smaller count is recorded; it is not passed off as a 40–60-trial test.

**High memorization accuracy does not prove correct X/y alignment:** random or incorrectly
assigned labels can also be memorized. The event-marker, trial-identity and saved-label checks
above provide the independent alignment checks. Poor memorization motivates inspecting
duplicates, constants, nonfinite values and contradictory labels.


In [ ]:
if RUN_MEMORIZATION:
    memory_family = "full" if "full" in subject_data["families"] else "bandpower"
    memorization_result = memorization_test(
        subject_data["families"][memory_family], y, trial_metadata, fixed_plan,
        RANDOM_SEED, subject_data["subject"], OUTPUT_ROOT)
    print("Memorization family:", memory_family)


## Phase 5–8: run the planned experiments

BASELINE_A evaluates linear SVM and logistic regression separately, as well as a procedure
that chooses between them by inner CV. It uses all 640 bandpower descriptors before constant
removal; the baseline does not use ANOVA to select an arbitrary smaller feature set.

For the current full feature family, the k sweep fixes one requested k per experiment and
tunes only classifier/C internally. Family comparisons choose k/classifier/C internally for
each separate family. `all` is optional because the instructor asks not to begin with thousands
of features. Its table row remains explicitly `not_run` while the option is disabled.

The nested procedure-selection experiment chooses a **single feature family**, k and
classifier/C inside each outer-training set. This evaluates the selection procedure without
using the outer results to decide a winner. It never merges all families into new descriptors.


In [ ]:
def evaluate_subject_stage(data, experiment, families, counts, classifiers=("linear_svm", "logistic"),
                           selector="anova", permutation_id=0):
    missing = set(families) - set(data["families"])
    if missing:
        print(f"{experiment}: pending existing feature exports {sorted(missing)}")
        return pd.DataFrame()
    result = evaluate_nested(
        F=data["F"], groups=data["family_groups"], y=data["y"],
        metadata=data["trial_metadata"], plan=data["plan"], subject=data["subject"],
        experiment=experiment, families=families, counts=counts, c_values=C_VALUES,
        seed=RANDOM_SEED, output_root=OUTPUT_ROOT, selector=selector,
        classifiers=classifiers, variance_threshold=VARIANCE_THRESHOLD,
        permutation_id=permutation_id, resume=RESUME, n_jobs=N_JOBS)
    ACTIVE_RUN_IDS.update(result.run_id.astype(str))
    return result


def run_planned_stages(data):
    if RUN_BASELINE_A:
        for classifier in ("linear_svm", "logistic"):
            evaluate_subject_stage(data, f"BASELINE_A_{classifier}", ["bandpower"], ["all"], [classifier])
        evaluate_subject_stage(data, "BASELINE_A_inner_selected", ["bandpower"], ["all"])

    if RUN_FEATURE_COUNT_SWEEP:
        counts = K_VALUES + (["all"] if INCLUDE_ALL_FEATURES_REFERENCE else [])
        for k in counts:
            evaluate_subject_stage(data, f"full_k_{k}", ["full"], [k])

    if RUN_FAMILY_COMPARISON:
        for family in FAMILY_NAMES:
            evaluate_subject_stage(data, f"family_{family}", [family], K_VALUES)

    if RUN_NESTED_PROCEDURE_SELECTION:
        evaluate_subject_stage(data, "NESTED_PROCEDURE_SELECTION", list(FAMILY_NAMES), K_VALUES)

    if RUN_MRMR:
        assert ANOVA_REVIEWED
        for selector in ("anova", "mrmr"):
            evaluate_subject_stage(data, f"full_{selector}_comparison", ["full"], MRMR_K_VALUES, selector=selector)

    if RUN_PERMUTATION_CONTROL:
        for permutation_id in range(1, N_PERMUTATIONS + 1):
            evaluate_subject_stage(data, "BASELINE_A_training_label_control", ["bandpower"], ["all"],
                                   permutation_id=permutation_id)


run_planned_stages(subject_data)


## Phase 9: training-only feature redundancy

This diagnostic uses only the first saved outer-training set. VarianceThreshold is fitted
there, and correlation is computed there. It counts all pairs above the declared absolute
correlation threshold using blocks, avoiding a full p × p allocation. Example pairs and
per-feature variances are saved separately.

The default variance threshold removes exact constants. Examine the variance distribution
before setting a positive near-zero threshold: original feature families can have different
units. Any chosen threshold remains inside every Pipeline fit. Correlated-pair inspection
does not automatically remove channels or globally prune features.


In [ ]:
if RUN_REDUNDANCY:
    training_indices = np.asarray(fixed_plan["folds"][0]["train"], dtype=int)
    redundancy_rows = []
    for family, matrix in subject_data["families"].items():
        feature_names = [f"{family}_{i}" for i in range(matrix.shape[1])]
        redundancy_rows.append(redundancy_diagnostic(
            matrix, feature_names, training_indices, OUTPUT_ROOT,
            subject_data["subject"], family, fixed_plan["folds"][0].get("fold", 1),
            correlation_threshold=CORRELATION_THRESHOLD,
            variance_threshold=VARIANCE_THRESHOLD))
    redundancy_summary = pd.DataFrame(redundancy_rows)
    display(redundancy_summary)


## Phase 10–11: later MRMR and shuffled-training-label controls

Enable MRMR only after setting `ANOVA_REVIEWED = True`; compare ANOVA/MRMR with the same
20/40/80/120 grid, saved folds and C grid. The library variant is disclosed above.

The permutation control shuffles only the outer-training labels within session, repeats
the entire inner search, and evaluates predictions against unchanged outer-test labels.
It saves controls separately via `permutation_id`; the real experiment is retained.
Training BA in these rows is measured against the shuffled labels and should not be
interpreted as real-label training accuracy.

Five repeats are an initial diagnostic. Their scores can vary around 25% and are not a
formal p-value. Consistently elevated null performance deserves investigation of leakage,
session-related structure and the control design; a single high score does not prove leakage.


## Phase 12: additional subjects

After reviewing the first subject, expand SUBJECTS and rerun this cell.
Each subject is loaded and evaluated independently; waveform/feature rows are not pooled
across people. Unrun subjects remain marked as not evaluated in the cohort summary.


In [ ]:
for subject in SUBJECTS[1:]:
    additional_data = prepare_subject(subject)
    run_planned_stages(additional_data)
    del additional_data


## Results: fold-level gaps, k curves and subject-wise OOF performance

Results from different run fingerprints must not be averaged together. The table below
filters to the currently prepared data/settings where possible and keeps run IDs visible.
Fold means are labeled as fold means. Subject test BA is computed from exact OOF predictions,
rather than assuming a mean of fold scores equals the pooled score.

`inner_selected_k_mode` is the most frequent k selected by inner CV across outer folds.
It is not a k chosen from outer-test accuracy. No automatic outer-score winner is returned.


In [ ]:
def current_results():
    path = OUTPUT_ROOT / "fold_results.csv"
    frame = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(frame) and ACTIVE_RUN_IDS:
        return frame.loc[frame.run_id.isin(ACTIVE_RUN_IDS)].copy()
    if len(frame):
        print("No stages were run in this execution: showing historical runs with separate run IDs.")
    return frame


results_df = current_results()
if len(results_df):
    real_results = results_df.loc[results_df.permutation_id == 0].copy()
    display(real_results[["run_id", "subject", "experiment", "feature_family", "requested_k",
                          "num_features", "classifier", "outer_fold", "train_balanced_accuracy",
                          "test_balanced_accuracy", "generalization_gap", "best_C"]])
    fold_summary = real_results.groupby(["run_id", "subject", "experiment"], as_index=False).agg(
        mean_train_BA=("train_balanced_accuracy", "mean"),
        mean_test_BA=("test_balanced_accuracy", "mean"),
        gap=("generalization_gap", "mean"),
        completed_outer_folds=("outer_fold", "nunique"),
    )
    display(fold_summary)
    save_csv_atomic(fold_summary, OUTPUT_ROOT / "fold_mean_summary.csv")
else:
    print("No completed experiment folds yet.")


In [ ]:
if len(results_df):
    sweep = results_df.loc[(results_df.permutation_id == 0) & results_df.experiment.str.startswith("full_k_")].copy()
    if len(sweep):
        k_table = sweep.groupby(["run_id", "subject", "requested_k"], as_index=False).agg(
            mean_train_BA=("train_balanced_accuracy", "mean"),
            mean_test_BA=("test_balanced_accuracy", "mean"),
            gap=("generalization_gap", "mean"),
            actual_selected_min=("num_features", "min"),
            actual_selected_max=("num_features", "max"),
        )
        k_table["status"] = "evaluated"
        if not INCLUDE_ALL_FEATURES_REFERENCE:
            k_table = pd.concat([k_table, pd.DataFrame([
                {"subject": subject, "requested_k": "all", "status": "not_run_optional_reference"}
                for subject in SUBJECTS])], ignore_index=True)
        display(k_table)
        save_csv_atomic(k_table, OUTPUT_ROOT / "k_comparison.csv")
        plotted = k_table.loc[k_table.status == "evaluated"].copy()
        plotted["numeric_k"] = pd.to_numeric(plotted.requested_k, errors="coerce")
        for subject, frame in plotted.dropna(subset=["numeric_k"]).groupby("subject"):
            if frame.numeric_k.duplicated().any():
                print("Multiple runs exist for this k/subject; select one configuration before plotting:", subject)
                continue
            frame = frame.sort_values("numeric_k")
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(frame.numeric_k, frame.mean_train_BA, "o-", label="Train BA")
            ax.plot(frame.numeric_k, frame.mean_test_BA, "o-", label="Outer-test BA")
            ax.axhline(.25, color="gray", linestyle="--", label="Chance")
            ax.set(xlabel="Requested ANOVA k", ylabel="Balanced accuracy", ylim=(0, 1),
                   title=f"Subject {subject:02d}: exploratory feature-count comparison")
            ax.legend()
            plt.tight_layout()
            plt.show()
    else:
        print("Enable RUN_FEATURE_COUNT_SWEEP after providing the original full feature export.")


In [ ]:
oof_path = OUTPUT_ROOT / "oof_predictions.csv"
subject_summary_rows = []
if len(results_df) and oof_path.exists():
    oof_df = pd.read_csv(oof_path)
    for (run_id, subject, experiment), fold_rows in results_df.loc[results_df.permutation_id == 0].groupby(
        ["run_id", "subject", "experiment"]):
        predictions = oof_df.loc[oof_df.run_id == run_id]
        if predictions.trial_id.duplicated().any():
            raise ValueError("Duplicate OOF trial predictions in one experiment.")
        run_manifest = json.loads((OUTPUT_ROOT / "run_manifests" / f"{run_id}.json").read_text())
        complete = (
            set(predictions.trial_id.astype(str)) == set(run_manifest["expected_trial_ids"])
            and len(predictions) == run_manifest["n_trials"]
            and set(fold_rows.outer_fold.astype(int)) == set(run_manifest["expected_fold_ids"])
        )
        subject_summary_rows.append({
            "run_id": run_id, "subject": subject, "experiment": experiment,
            "Train_BA_fold_mean": fold_rows.train_balanced_accuracy.mean(),
            "Test_BA_OOF": balanced_accuracy_score(predictions.y_true, predictions.y_pred) if complete else np.nan,
            "inner_selected_k_mode": str(fold_rows.requested_k.mode().iloc[0]),
            "OOF_trials": len(predictions), "outer_folds_completed": fold_rows.outer_fold.nunique(),
            "status": "complete" if complete else "incomplete_do_not_report_as_final",
        })
    subject_summary = pd.DataFrame(subject_summary_rows)
    display(subject_summary)
    save_csv_atomic(subject_summary, OUTPUT_ROOT / "subject_summary.csv")
    cohort_status = pd.DataFrame({"subject": range(1, 11)})
    cohort_status["status"] = np.where(cohort_status.subject.isin(subject_summary.subject), "has_results", "not_evaluated")
    display(cohort_status)


In [ ]:
if len(results_df):
    controls = results_df.loc[results_df.permutation_id > 0]
    if len(controls) and oof_path.exists():
        permutation_rows = []
        for run_id, frame in controls.groupby("run_id"):
            predictions = oof_df.loc[oof_df.run_id == run_id]
            run_manifest = json.loads((OUTPUT_ROOT / "run_manifests" / f"{run_id}.json").read_text())
            if (len(predictions) != run_manifest["n_trials"]
                    or set(predictions.trial_id.astype(str)) != set(run_manifest["expected_trial_ids"])):
                continue
            permutation_rows.append({"run_id": run_id, "subject": int(frame.subject.iloc[0]),
                                     "permutation_id": int(frame.permutation_id.iloc[0]),
                                     "test_BA_OOF": balanced_accuracy_score(predictions.y_true, predictions.y_pred)})
        permutation_summary = pd.DataFrame(permutation_rows)
        display(permutation_summary)
        if permutation_summary.empty:
            print("No complete permutation repetitions yet.")
        else:
            display(permutation_summary.groupby("subject")["test_BA_OOF"].agg(["mean", "std", "count"]))
        save_csv_atomic(permutation_summary, OUTPUT_ROOT / "permutation_summary.csv")
        if len(permutation_summary):
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.hist(permutation_summary.test_BA_OOF, bins=min(10, len(permutation_summary)), edgecolor="white")
            ax.axvline(.25, color="red", linestyle="--", label="25% chance")
            ax.set(xlabel="Outer OOF BA with shuffled training labels", ylabel="Controls", title="Diagnostic null scores")
            ax.legend()
            plt.tight_layout()
            plt.show()


## Phase 13–14: freeze the procedure, then plan channel reduction

Review inner selection, convergence, outer BA/gaps, subject variability and label controls.
A lower training BA alone is not success; the complete nested evaluation needs to support
useful generalization. The instructor's 0.35–0.50 test range is an aspiration, not an acceptance
threshold or a promised outcome.

Enable RUN_NESTED_PROCEDURE_SELECTION to evaluate the family/k/classifier selection procedure
using inner CV. If you later freeze it, freeze its extraction definitions, candidate families,
selection strategy, C/k grids, seeds, metrics and saved split signatures—not a global k/C
picked from outer-test curves. Per-fold k/C selection still occurs only in training.

The code below saves a protocol manifest only after you explicitly set FREEZE_PROCEDURE.
It does not begin 64/32/16-channel evaluation. Later use your previously saved physical
channel configurations; no new channel subsets are generated here. Feature exports for those
channel counts must come from the same existing extraction procedure on the fixed channels.

Because earlier outer scores have informed development, these comparisons are exploratory
until independently confirmed. Reusing frozen folds improves comparability but does not
make repeated development on their outcomes an untouched final test.


In [ ]:
if FREEZE_PROCEDURE:
    if not ANOVA_REVIEWED:
        raise ValueError("Review the ANOVA diagnostics before freezing the procedure.")
    if not RUN_NESTED_PROCEDURE_SELECTION:
        raise ValueError("Evaluate the inner-selected procedure before freezing it.")
    if set(FAMILY_NAMES) - set(subject_data["families"]):
        raise ValueError("Provide and evaluate all original feature families before freezing this full comparison.")
    completed_subjects = {
        int(row["subject"]) for row in subject_summary_rows
        if row["experiment"] == "NESTED_PROCEDURE_SELECTION" and row["status"] == "complete"
        and row["run_id"] in ACTIVE_RUN_IDS
    }
    if not set(SUBJECTS).issubset(completed_subjects):
        raise ValueError("Every requested subject needs a complete current-run outer OOF result before freezing.")
    manifest = {
        "core_version": CORE_VERSION,
        "subjects_requested": SUBJECTS,
        "channel_count_at_debugging": 128,
        "evaluation": "subject-dependent frozen nested CV",
        "seed": RANDOM_SEED,
        "metric": "balanced_accuracy",
        "input_mode": INPUT_MODE,
        "action_window_mne": list(ACTION_WINDOW_MNE), "endpoint": ACTION_ENDPOINT,
        "feature_families": list(subject_data["families"]),
        "feature_extraction": "fixed PSD control plus existing raw per-trial descriptor exports",
        "selector": "anova", "k_grid": K_VALUES, "C_grid": C_VALUES,
        "classifiers": ["linear_svm", "logistic"],
        "variance_threshold": VARIANCE_THRESHOLD,
        "selection": "inner CV; outer results do not select the winner",
        "fixed_split_source": str(SAVED_SPLITS_ROOT),
        "fixed_channel_configurations": "supply the previously saved 128/64/32/16 configurations later",
        "channel_reduction_started": False,
    }
    save_json_atomic(manifest, OUTPUT_ROOT / "frozen_baseline_procedure.json")
    print("Saved procedure manifest. Channel reduction remains deferred.")


## Export contracts for the existing inputs and feature families

**Frozen plans:** copy existing split JSON files into SAVED_SPLITS_ROOT. Files can keep their
original names as long as they start with `sub-01`, `sub-02`, etc. The loader searches recursively.
It checks exact IDs/labels and keeps all saved indices. It does not silently regenerate a plan.

**Exact model input, if needed:** create `sub-01.npz` in EXISTING_INPUT_ROOT containing `X`, `y`,
`sessions`, `trial_ids`, `ch_names`, `times`. X is your unchanged action-window EEG array.
`sub-01.json` declares `sfreq`, `units` (`volts` or `microvolts`), `inner_speech_only: true`,
and the original preprocessing/action-window provenance. Trial IDs must match the saved plans.

**Existing descriptors:** create `sub-01.npz` in FEATURE_EXPORT_ROOT containing `trial_ids`, `y`,
and raw matrices named `wavelet`, `time_domain`, `regional`, `full`. Each available matrix is
[trials, features]. `full` is the current original full set, not a newly invented fusion.
The matching `sub-01.json` must declare:

```json
{
  "fit_scope": "stateless_per_trial",
  "channel_count": 128,
  "model_input_sha256": "the canonical input fingerprint printed by prepare_subject",
  "extractor_version": "your existing extractor's version/hash"
}
```

Compute the fingerprint using `array_digest` on the canonical float32 input in volts,
before saved-plan reordering. Do not relabel unrelated exports with a matching fingerprint.
Only declare stateless per-trial features after verifying no transform was globally fitted.
Fitted regional/spatial PCA or global feature normalization cannot be exported as a safe raw
matrix; their fit must be moved inside CV first.

Use the following optional export helper **in the environment where your original raw feature
matrices already exist**. It does not compute or replace the current wavelet features.


In [ ]:
def export_existing_descriptors(export_root, subject, canonical_X_volts, trial_ids, y,
                                feature_matrices, extractor_version):
    """Call only for verified stateless, pre-selection, per-trial feature matrices."""
    allowed = {"wavelet", "time_domain", "regional", "full"}
    if not set(feature_matrices).issubset(allowed):
        raise ValueError("Use the documented existing feature-family names.")
    trial_ids = np.asarray(trial_ids, dtype=str)
    labels = np.asarray(y, dtype=int)
    canonical_X_volts = np.asarray(canonical_X_volts, dtype=np.float32)
    if (canonical_X_volts.ndim != 3 or canonical_X_volts.shape[1] != 128
            or len(canonical_X_volts) != len(labels)):
        raise ValueError("Canonical EEG must align with labels and contain all 128 channels.")
    if len(trial_ids) != len(labels) or len(set(trial_ids)) != len(labels):
        raise ValueError("Trial IDs and labels must align one-to-one.")
    for name, matrix in feature_matrices.items():
        if np.asarray(matrix).ndim != 2 or len(matrix) != len(labels) or not np.isfinite(matrix).all():
            raise ValueError(f"Invalid raw feature matrix: {name}")
    prefix = Path(export_root) / f"sub-{int(subject):02d}"
    prefix.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(prefix.with_suffix(".npz"), trial_ids=trial_ids, y=labels, **feature_matrices)
    save_json_atomic({"fit_scope": "stateless_per_trial", "channel_count": 128,
                      "model_input_sha256": array_digest(np.asarray(canonical_X_volts, dtype=np.float32)),
                      "extractor_version": extractor_version}, prefix.with_suffix(".json"))


# Example calling structure, after binding your existing variables:
# export_existing_descriptors(
#     FEATURE_EXPORT_ROOT, subject=1, canonical_X_volts=your_existing_X_in_volts,
#     trial_ids=your_original_trial_ids, y=your_original_y,
#     feature_matrices={"wavelet": your_raw_wavelet_features,
#                       "time_domain": your_raw_time_features,
#                       "regional": your_raw_regional_features,
#                       "full": your_raw_full_features},
#     extractor_version="existing-version-or-source-hash",
# )


## Files saved and variables for later code

`fold_results.csv` is the persistent main dataframe. `oof_predictions.csv` retains exact
test predictions. Other folders retain audits, run manifests, inner CV results, convergence
warnings, memorization checks and training-only redundancy diagnostics. Summary CSVs are
derived views and preserve separate run IDs.

Canonical variables: `X`, `y`, `trial_metadata`, `ch_names`, `sfreq`, `times`, `fixed_plan`,
`subject_data`, `results_df`, `OUTPUT_ROOT`. Here **X contains the unchanged action interval**,
unlike the earlier EDA starter's full 4.5-second epoch. Units remain volts.

Inspect the phase flags in the settings cell and progress one experiment at a time.
No results are prefilled and no improved accuracy is claimed before running your data.

API references: [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html),
[GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html),
[nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html),
[mrmr-selection implementation](https://github.com/smazzanti/mrmr).
